# Math Requirements for Molecule Analysis

This notebook turns the math plan into a concrete learning map tied to the datasets in `data/`.

Covered topics:
- Algebra
- Linear Algebra
- Calculus
- Multivariable Calculus
- Statistics

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == 'notebooks' else cwd


PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / 'data'

delaney = pd.read_csv(DATA_DIR / 'delaney-processed.csv')
bbbp = pd.read_csv(DATA_DIR / 'BBBP.csv')

numeric_features = [
    'ESOL predicted log solubility in mols per litre',
    'Minimum Degree',
    'Molecular Weight',
    'Number of H-Bond Donors',
    'Number of Rings',
    'Number of Rotatable Bonds',
    'Polar Surface Area',
]
target = 'measured log solubility in mols per litre'

print('Delaney shape:', delaney.shape)
print('BBBP shape:', bbbp.shape)

## 1. Algebra

Core requirements:
- linear and affine functions
- logarithms and exponentials
- ratios and rates
- systems of equations
- threshold-based decision rules

Why this matters here:
- property targets like solubility are often already log-scaled
- linear models combine weighted feature sums
- binary classification turns continuous scores into thresholded decisions

In [ ]:
algebra_demo = delaney[[
    'Molecular Weight',
    'Number of Rotatable Bonds',
    target,
]].copy()
algebra_demo['simple_linear_score'] = (
    -0.01 * algebra_demo['Molecular Weight']
    -0.20 * algebra_demo['Number of Rotatable Bonds']
    + 1.0
)
algebra_demo['predicted_soluble'] = (algebra_demo['simple_linear_score'] > -2.0).astype(int)
display(algebra_demo.head(10))

## 2. Linear Algebra

Core requirements:
- vectors and matrices
- matrix multiplication
- dot products and projections
- covariance matrices
- eigenvalues and eigenvectors
- PCA intuition

Why this matters here:
- descriptor tables are feature matrices
- model training is mostly vectorized matrix algebra
- dimensionality reduction is useful for wide molecular descriptors

In [ ]:
X = delaney[numeric_features].to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
covariance = np.cov(X_scaled, rowvar=False)
covariance_df = pd.DataFrame(covariance, index=numeric_features, columns=numeric_features)
display(covariance_df.round(3))

pca = PCA(n_components=2, random_state=42)
components = pca.fit_transform(X_scaled)
pca_df = pd.DataFrame({'PC1': components[:, 0], 'PC2': components[:, 1], 'target': delaney[target]})

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='target', palette='viridis', s=40)
plt.title('PCA Projection of Delaney Features')
plt.show()

print('Explained variance ratio:', pca.explained_variance_ratio_)

## 3. Calculus

Core requirements:
- derivatives
- slope and local sensitivity
- chain rule
- optimization of loss functions
- gradient descent intuition

Why this matters here:
- training minimizes a loss
- model parameters move in the direction of negative gradients

In [ ]:
x = delaney['Molecular Weight'].to_numpy()[:200]
y = delaney[target].to_numpy()[:200]

def mse_for_weight(weight: float) -> float:
    predictions = weight * x
    return float(np.mean((predictions - y) ** 2))

weights = np.linspace(-0.05, 0.05, 200)
losses = np.array([mse_for_weight(weight) for weight in weights])

plt.figure(figsize=(8, 5))
plt.plot(weights, losses)
plt.title('Single-Parameter Loss Curve')
plt.xlabel('weight')
plt.ylabel('MSE')
plt.show()

## 4. Multivariable Calculus

Core requirements:
- gradients
- partial derivatives
- Jacobian and Hessian intuition
- multi-parameter optimization

Why this matters here:
- realistic models have many parameters
- multi-task models optimize a shared parameter surface

In [ ]:
subset = delaney[['Molecular Weight', 'Polar Surface Area', target]].dropna().head(200).copy()
X_small = subset[['Molecular Weight', 'Polar Surface Area']].to_numpy()
y_small = subset[target].to_numpy()

weights = np.zeros(2)
bias = 0.0

pred = X_small @ weights + bias
error = pred - y_small
grad_w = (2 / len(X_small)) * X_small.T @ error
grad_b = float((2 / len(X_small)) * error.sum())

print('Gradient with respect to weights:', grad_w)
print('Gradient with respect to bias:', grad_b)

## 5. Statistics

Core requirements:
- descriptive statistics
- distributions
- correlation and covariance
- class balance
- sampling and splits
- confidence intervals and uncertainty
- evaluation metrics

Why this matters here:
- small clean datasets and sparse multi-task datasets behave very differently
- bad splits or ignored imbalance can make results misleading

In [ ]:
summary_stats = delaney[numeric_features + [target]].describe().T
display(summary_stats)

plt.figure(figsize=(8, 5))
sns.histplot(delaney[target], bins=30, kde=True)
plt.title('Distribution of Measured Log Solubility')
plt.show()

plt.figure(figsize=(6, 4))
sns.countplot(data=bbbp, x='p_np')
plt.title('BBBP Class Balance')
plt.xlabel('p_np')
plt.show()

## Minimum Readiness Checklist

A learner is ready for the first model notebooks when they can:
- explain a weighted sum and threshold for binary classification
- interpret a feature matrix and a target vector
- describe a loss function and what a gradient means
- reason about class balance and train/test splits
- read PCA output and understand why dimensionality reduction may help